In [ ]:
import torch

# Создаем пример с градиентами
N = 40
matrices = torch.randn(N, 2, 2, requires_grad=True)
print("Матрицы требуют градиенты:", matrices.requires_grad)

# Способ 1: Чистый тензорный подход с torch.cumulative_product аналогом
def sequential_multiply_tensor_only(matrices):
    """Полностью векторизованный подход без списков"""
    result = matrices[0]  # Начинаем с первой матрицы
    for i in range(1, matrices.shape[0]):
        result = torch.matmul(result, matrices[i])
    return result

# Способ 2: Использование torch.unbind (избегает явного индексирования)
def sequential_multiply_unbind(matrices):
    """Использует unbind вместо списка"""
    unbinded = torch.unbind(matrices, dim=0)  # Разбивает по первому измерению
    result = unbinded[0]
    for matrix in unbinded[1:]:
        result = torch.matmul(result, matrix)
    return result

# Способ 3: Functional reduce с torch.unbind
def sequential_multiply_functional(matrices):
    from functools import reduce
    return reduce(torch.matmul, torch.unbind(matrices, dim=0))

# Способ 4: Рекурсивная функция с сохранением тензорной структуры
def sequential_multiply_recursive_tensor(matrices):
    if matrices.shape[0] == 1:
        return matrices[0]
    elif matrices.shape[0] == 2:
        return torch.matmul(matrices[0], matrices[1])
    else:
        mid = matrices.shape[0] // 2
        left = sequential_multiply_recursive_tensor(matrices[:mid])
        right = sequential_multiply_recursive_tensor(matrices[mid:])
        return torch.matmul(left, right)

# Способ 5: Multi_dot с проверкой градиентов
def sequential_multiply_multidot_safe(matrices):
    matrix_list = [matrices[i] for i in range(matrices.shape[0])]
    return torch.linalg.multi_dot(matrix_list)

# Тестируем все методы
print("\nТестирование сохранения градиентов:")

methods = [
    ("Тензорный цикл", sequential_multiply_tensor_only),
    ("Unbind", sequential_multiply_unbind),
    ("Functional", sequential_multiply_functional),
    ("Рекурсивный", sequential_multiply_recursive_tensor),
    ("Multi_dot", sequential_multiply_multidot_safe)
]

results = {}
for name, method in methods:
    result = method(matrices)
    results[name] = result
    print(f"{name}: requires_grad = {result.requires_grad}")

# Проверяем, что все результаты одинаковые
base_result = results["Тензорный цикл"]
for name, result in results.items():
    print(f"{name} == базовый: {torch.allclose(result, base_result)}")

# Тестируем обратное распространение
print("\nТест обратного распространения:")
for name, method in methods:
    matrices_copy = torch.randn(N, 2, 2, requires_grad=True)
    result = method(matrices_copy)

    # Вычисляем скаляр для backward
    scalar_loss = result.sum()
    scalar_loss.backward()

    print(f"{name}: градиенты вычислены = {matrices_copy.grad is not None}")
    if matrices_copy.grad is not None:
        print(f"  Норма градиента: {matrices_copy.grad.norm().item():.6f}")

# Полный benchmark всех методов
def full_benchmark(N=8000, with_gradients=True):
    import time

    print(f"\nПолный бенчмарк всех методов (N={N}, gradients={with_gradients}):")
    print("-" * 60)

    methods = [
        ("Тензорный цикл", sequential_multiply_tensor_only),
        ("Unbind", sequential_multiply_unbind),
        ("Functional", sequential_multiply_functional),
        ("Рекурсивный", sequential_multiply_recursive_tensor),
        ("Multi_dot", sequential_multiply_multidot_safe)
    ]

    results = {}

    for name, method in methods:
        times = []

        # Прогреваем
        for _ in range(3):
            matrices_warmup = torch.randn(10, 2, 2, requires_grad=with_gradients)
            _ = method(matrices_warmup)

        # Основные измерения
        for run in range(5):
            matrices_test = torch.randn(N, 2, 2, requires_grad=with_gradients)

            start = time.perf_counter()
            result = method(matrices_test)

            if with_gradients:
                loss = result.sum()
                loss.backward()

            end = time.perf_counter()
            times.append(end - start)

        avg_time = sum(times) / len(times)
        std_time = (sum((t - avg_time) ** 2 for t in times) / len(times)) ** 0.5
        results[name] = (avg_time, std_time, result if 'result' in locals() else None)

        print(f"{name:15}: {avg_time:.6f} ± {std_time:.6f} сек")

    # Проверяем корректность результатов
    print("\nПроверка корректности:")
    first_result = None
    for name, (_, _, result) in results.items():
        if result is not None:
            if first_result is None:
                first_result = result
            else:
                is_close = torch.allclose(result, first_result, atol=1e-5)
                print(f"{name:15}: {'✓' if is_close else '✗'}")

    # Определяем победителя
    fastest = min(results.items(), key=lambda x: x[1][0])
    print(f"\n🏆 Самый быстрый: {fastest[0]} ({fastest[1][0]:.6f} сек)")

    return results

print("\n" + "=" * 70)
print("БЕНЧМАРК С ГРАДИЕНТАМИ")
print("=" * 70)

for N in [10000]:
    full_benchmark(N, with_gradients=True)

Матрицы требуют градиенты: True

Тестирование сохранения градиентов:
Тензорный цикл: requires_grad = True
Unbind: requires_grad = True
Functional: requires_grad = True
Рекурсивный: requires_grad = True
Multi_dot: requires_grad = True
Тензорный цикл == базовый: True
Unbind == базовый: True
Functional == базовый: True
Рекурсивный == базовый: True
Multi_dot == базовый: True

Тест обратного распространения:
Тензорный цикл: градиенты вычислены = True
  Норма градиента: 4432.126465
Unbind: градиенты вычислены = True
  Норма градиента: 1.208364
Functional: градиенты вычислены = True
  Норма градиента: 3069.103027
Рекурсивный: градиенты вычислены = True
  Норма градиента: 236.097931
Multi_dot: градиенты вычислены = True
  Норма градиента: 231.851837

БЕНЧМАРК С ГРАДИЕНТАМИ

Полный бенчмарк всех методов (N=10000, gradients=True):
------------------------------------------------------------
Тензорный цикл : 0.391832 ± 0.025923 сек
Unbind         : 0.157408 ± 0.029561 сек
Functional     : 0.13896